# Step 2: Train Fraud Detection Model

**Goal**: Train simple binary classifier

- Load data from bronze table
- Create 4 simple features
- Train XGBoost
- Evaluate and save

In [0]:
%pip install xgboost

In [0]:
# Imports
from pyspark.sql.functions import hour, dayofweek, month
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import pickle
from datetime import datetime

print("✓ Imports loaded")

In [0]:
# Load data
print("Loading data...")
df = spark.read.table("workspace.bronze.fraud_data")
print(f"✓ Loaded {df.count():,} transactions")

In [0]:
# Create features
df_features = df.select(
    hour("timestamp").alias("hour"),
    dayofweek("timestamp").alias("day_of_week"),
    month("timestamp").alias("month"),
    "amount_inr",
    "is_fraud"
)

df_pandas = df_features.toPandas()
print(f"✓ Features: {df_pandas.shape}")
print(f"✓ Fraud: {df_pandas['is_fraud'].sum()} ({df_pandas['is_fraud'].sum()/len(df_pandas)*100:.2f}%)")

In [0]:
# Split data
X = df_pandas[['hour', 'day_of_week', 'month', 'amount_inr']]
y = df_pandas['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Train: {len(X_train):,} samples")
print(f"✓ Test: {len(X_test):,} samples")

In [0]:
# Train model
model = xgb.XGBClassifier(
    objective='binary:logistic',
    max_depth=4,
    learning_rate=0.1,
    n_estimators=100,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("✓ Trained")
print(f"Accuracy: {(y_pred == y_test).mean():.4f}")

In [0]:
# Evaluate
print("\n" + "="*50)
print("EVALUATION")
print("="*50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'], zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"              Predicted")
print(f"            Legit  Fraud")
print(f"Actual Legit {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"       Fraud {cm[1,0]:5d}  {cm[1,1]:5d}")

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"\nROC-AUC: {roc_auc:.4f}")
except:
    print("\nROC-AUC: N/A")

In [0]:
# Save model
model_artifact = {
    'model': model,
    'features': ['hour', 'day_of_week', 'month', 'amount_inr'],
    'trained_at': datetime.now()
}

model_bytes = pickle.dumps(model_artifact)
model_df = spark.createDataFrame([(
    'fraud_model_v1',
    model_bytes,
    datetime.now()
)], ['model_name', 'model_binary', 'saved_at'])

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.models")
model_df.write.format('delta').mode('overwrite').saveAsTable('workspace.models.fraud_model')

print("✓ Saved to workspace.models.fraud_model")